# Email Spam Detection — Enhanced NLP + ML/DL/BERT Pipeline

This notebook builds spam-classifier project with a much richer
NLP pipeline and compares three tiers of models:

1. **Classical ML** — Multinomial Naive Bayes, Linear SVM, Logistic Regression (on TF-IDF n-grams)
2. **Neural Network** — an Embedding + BiLSTM Keras model (learns its own representations)
3. **Transformer (BERT)** — fine-tuned `bert-base-uncased` via HuggingFace `transformers`


## 1. Setup & Imports

In [ ]:
# Core
import re
import string
import pickle
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# NLP
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk import pos_tag

# Feature extraction / classical ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
                              accuracy_score, precision_score, recall_score,
                              roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 2. Load Data & Exploratory Analysis
Same source data as before (`emails.csv`), but we drop duplicates and strip the
constant `"Subject:"` prefix before any NLP so it doesn't pollute the vocabulary.

In [ ]:
df = pd.read_csv('emails.csv')
print(df.shape)
df.head()

In [ ]:
df = df.drop_duplicates(keep='last').reset_index(drop=True)
df['text'] = df['text'].str.replace(r'^Subject:\s*', '', regex=True)
print('After dedup:', df.shape)
df['spam'].value_counts(normalize=True)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
df['spam'].value_counts().plot(kind='pie', autopct='%.1f%%', labels=['Ham', 'Spam'], ax=ax[0])
ax[0].set_ylabel('')
ax[0].set_title('Class balance')
sns.countplot(x='spam', data=df, ax=ax[1])
ax[1].set_xticklabels(['Ham', 'Spam'])
ax[1].set_title('Class counts')
plt.tight_layout()
plt.show()

In [ ]:
df['char_len'] = df['text'].str.len()
plt.figure(figsize=(8, 4))
sns.histplot(data=df, x='char_len', hue='spam', bins=60, log_scale=(False, True))
plt.title('Message length distribution: ham vs spam')
plt.xlim(0, 5000)
plt.show()

## 3. Enhanced NLP Preprocessing

The original pipeline only did punctuation stripping + Porter stemming. Stemming
is crude (`"organization"` → `"organ"`); **lemmatization** uses vocabulary +
POS tags to return real dictionary forms (`"running"` → `"run"`, not `"run"` by
chance-truncation). We POS-tag each token first so the lemmatizer knows whether
a word is a noun/verb/adjective/adverb — this is the "backoff"-style refinement
you asked about (falling back to the base/dictionary form of a word rather than
a truncated stem).

Steps:
1. Lowercase
2. Strip everything except letters and whitespace (removes digits, punctuation, stray symbols)
3. Tokenize (`word_tokenize`)
4. Remove stopwords and very short tokens (len <= 2)
5. POS-tag remaining tokens
6. Lemmatize using the POS tag
7. Rejoin into a cleaned string


In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()  # kept for comparison only

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    if tag.startswith('V'):
        return wordnet.VERB
    if tag.startswith('R'):
        return wordnet.ADV
    return wordnet.NOUN  # default / backoff

def clean_and_lemmatize(text: str) -> str:
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = [t for t in word_tokenize(text) if t not in stop_words and len(t) > 2]
    tagged = pos_tag(tokens)
    lemmas = [lemmatizer.lemmatize(w, get_wordnet_pos(p)) for w, p in tagged]
    return ' '.join(lemmas)

# Sanity check: stemming vs lemmatization on one example
sample = df['text'].iloc[0][:200]
print('RAW       :', sample)
print()
print('LEMMATIZED:', clean_and_lemmatize(sample))


In [ ]:
t0 = time.time()
df['clean_text'] = df['text'].apply(clean_and_lemmatize)
print(f'Preprocessed {len(df)} emails in {time.time()-t0:.1f}s')
df[['text', 'clean_text', 'spam']].head()

In [ ]:
# Optional: word clouds to sanity-check vocabulary per class
try:
    from wordcloud import WordCloud
    fig, ax = plt.subplots(1, 2, figsize=(14, 6))
    for i, label in enumerate([0, 1]):
        text_blob = ' '.join(df.loc[df.spam == label, 'clean_text'])
        wc = WordCloud(width=600, height=400, background_color='white').generate(text_blob)
        ax[i].imshow(wc)
        ax[i].axis('off')
        ax[i].set_title('Ham' if label == 0 else 'Spam')
    plt.show()
except ImportError:
    print('pip install wordcloud to see this visualization (optional).')


## 4. Feature Extraction — TF-IDF with n-grams

`CountVectorizer` (raw counts) over-weights very frequent but low-signal words.
`TfidfVectorizer` down-weights words that appear in most documents and
up-weights ones that are distinctive to a class. Adding **bigrams**
(`ngram_range=(1,2)`) lets the model catch short spam phrases like
`"click here"` or `"free offer"` that unigrams alone can't represent.

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_text'], df['spam'], test_size=0.2, stratify=df['spam'], random_state=RANDOM_STATE
)

tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=6000, min_df=2, sublinear_tf=True)
X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)
print('Train shape:', X_train.shape, '| Test shape:', X_test.shape)

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)


## 5. Tier 1 — Classical ML Models

`GaussianNB` (used in the original notebook) assumes continuous, roughly
Gaussian features — it's a poor theoretical fit for sparse word-count/TF-IDF
data. `MultinomialNB` is the standard, correct choice for text. We also swap
plain `SVC` for `LinearSVC`, which scales far better on high-dimensional sparse
text data, and add stratified cross-validation plus a full metric suite
(precision / recall / F1 / ROC-AUC), since accuracy alone is misleading on an
imbalanced 76/24 dataset.

In [ ]:
def evaluate(name, model, X_tr, y_tr, X_te, y_te, use_decision_function=True):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    if use_decision_function and hasattr(model, 'decision_function'):
        y_score = model.decision_function(X_te)
    elif hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(X_te)[:, 1]
    else:
        y_score = y_pred

    cv_f1 = cross_val_score(model, X_tr, y_tr, cv=5, scoring='f1').mean()

    metrics = {
        'model': name,
        'accuracy': accuracy_score(y_te, y_pred),
        'precision': precision_score(y_te, y_pred),
        'recall': recall_score(y_te, y_pred),
        'f1': f1_score(y_te, y_pred),
        'roc_auc': roc_auc_score(y_te, y_score),
        'cv_f1_mean': cv_f1,
    }
    print(f"--- {name} ---")
    print(classification_report(y_te, y_pred, target_names=['Ham', 'Spam']))

    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ConfusionMatrixDisplay.from_predictions(y_te, y_pred, display_labels=['Ham', 'Spam'], ax=ax[0])
    ax[0].set_title(f'{name} — Confusion Matrix')
    RocCurveDisplay.from_predictions(y_te, y_score, ax=ax[1])
    ax[1].set_title(f'{name} — ROC Curve')
    plt.tight_layout()
    plt.show()

    return model, metrics

results = []


In [ ]:
nb_model, m = evaluate('MultinomialNB', MultinomialNB(), X_train, y_train, X_test, y_test,
                        use_decision_function=False)
results.append(m)

In [ ]:
svm_model, m = evaluate('LinearSVC', LinearSVC(class_weight='balanced', random_state=RANDOM_STATE),
                         X_train, y_train, X_test, y_test)
results.append(m)

In [ ]:
lr_model, m = evaluate('LogisticRegression',
                        LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
                        X_train, y_train, X_test, y_test)
results.append(m)

In [ ]:
# Pick the best classical model by F1 and persist it (mirrors the original save-to-pickle step)
best_classical_row = max(results, key=lambda r: r['f1'])
best_classical_name = best_classical_row['model']
best_classical_model = {'MultinomialNB': nb_model, 'LinearSVC': svm_model, 'LogisticRegression': lr_model}[best_classical_name]
print('Best classical model:', best_classical_name)

with open('spam_classifier.pkl', 'wb') as f:
    pickle.dump(best_classical_model, f)


## 6. Tier 2 — Neural Network (Embedding + BiLSTM)

Classical ML treats words as independent columns. A neural network with a
trainable **embedding layer** learns dense word representations and, with an
**LSTM**, can pick up word *order* (e.g. "not a scam" vs "a scam"), which
bag-of-words approaches cannot.

> **Requires**: `tensorflow` (`pip install tensorflow`). Not run automatically
> in this environment — run this section locally or on Colab/Antigravity where
> TensorFlow is available.

In [ ]:
# pip install tensorflow  # uncomment if not installed
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

MAX_VOCAB = 10000
MAX_LEN = 200

tok = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
tok.fit_on_texts(X_train_text)

X_train_seq = pad_sequences(tok.texts_to_sequences(X_train_text), maxlen=MAX_LEN, padding='post', truncating='post')
X_test_seq = pad_sequences(tok.texts_to_sequences(X_test_text), maxlen=MAX_LEN, padding='post', truncating='post')

nn_model = Sequential([
    Embedding(input_dim=MAX_VOCAB, output_dim=128, input_length=MAX_LEN),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.4),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid'),
])
nn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
nn_model.summary()


In [ ]:
es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
history = nn_model.fit(
    X_train_seq, y_train,
    validation_split=0.1,
    epochs=15,
    batch_size=32,
    class_weight={0: 1.0, 1: (y_train == 0).sum() / (y_train == 1).sum()},  # handle imbalance
    callbacks=[es],
)


In [ ]:
nn_probs = nn_model.predict(X_test_seq).ravel()
nn_preds = (nn_probs > 0.5).astype(int)

print(classification_report(y_test, nn_preds, target_names=['Ham', 'Spam']))
results.append({
    'model': 'BiLSTM-NN',
    'accuracy': accuracy_score(y_test, nn_preds),
    'precision': precision_score(y_test, nn_preds),
    'recall': recall_score(y_test, nn_preds),
    'f1': f1_score(y_test, nn_preds),
    'roc_auc': roc_auc_score(y_test, nn_probs),
    'cv_f1_mean': np.nan,  # CV not typically done for deep nets this way
})

nn_model.save('spam_bilstm.keras')
import pickle as pkl
with open('nn_tokenizer.pkl', 'wb') as f:
    pkl.dump(tok, f)


## 7. Tier 3 — Fine-tuned BERT

BERT produces **contextual** embeddings — the same word gets a different
vector depending on its sentence, which handles negation, sarcasm and
polysemy far better than bag-of-words or a shallow BiLSTM. This is the most
expensive tier (needs a GPU for reasonable training time).

> **Requires**: `pip install transformers torch datasets accelerate`.
> Also not run automatically here — the HuggingFace model hub is not
> reachable from this sandboxed tool environment. Run this section on
> Colab, a local GPU machine, or inside Antigravity (see §9).

In [ ]:
# pip install transformers torch datasets accelerate scikit-learn
import torch
from torch.utils.data import Dataset
from transformers import (BertTokenizerFast, BertForSequenceClassification,
                           TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score as acc_fn, f1_score as f1_fn

MODEL_NAME = 'bert-base-uncased'
bert_tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

class EmailDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=max_len)
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

# NOTE: use the ORIGINAL (non-lemmatized) text for BERT — it has its own
# subword tokenizer and benefits from natural sentence structure/punctuation.
raw_train_text, raw_test_text, _, _ = train_test_split(
    df['text'], df['spam'], test_size=0.2, stratify=df['spam'], random_state=RANDOM_STATE
)

train_ds = EmailDataset(raw_train_text, y_train, bert_tokenizer)
test_ds = EmailDataset(raw_test_text, y_test, bert_tokenizer)

bert_model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': acc_fn(labels, preds), 'f1': f1_fn(labels, preds)}

training_args = TrainingArguments(
    output_dir='./bert_spam_ckpt',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_steps=50,
    learning_rate=2e-5,
    weight_decay=0.01,
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

trainer.train()


In [ ]:
bert_eval = trainer.evaluate()
print(bert_eval)

results.append({
    'model': 'BERT-fine-tuned',
    'accuracy': bert_eval.get('eval_accuracy', np.nan),
    'precision': np.nan,
    'recall': np.nan,
    'f1': bert_eval.get('eval_f1', np.nan),
    'roc_auc': np.nan,
    'cv_f1_mean': np.nan,
})

bert_model.save_pretrained('./bert_spam_model')
bert_tokenizer.save_pretrained('./bert_spam_model')


## 8. Model Comparison

In [ ]:
results_df = pd.DataFrame(results).set_index('model')
display(results_df.round(4))

results_df[['accuracy', 'f1', 'roc_auc']].dropna(how='all').plot(kind='bar', figsize=(9, 5))
plt.title('Model comparison')
plt.ylabel('Score')
plt.ylim(0.8, 1.01)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()
